# AUAF — Stage 4.5, Part 2 (Corrected v3): Train-only Preprocessing and Model Factory

**Purpose:** extend the verified Part 1 data loader with explicit reviewer-aligned feature sets, train-only preprocessing, participant-availability audits, and a canonical model factory.

**This part does not train or tune models.** It does not change the frozen cohort rows, labels, or split assignments. It does not overwrite previous checkpoints or results.

The notebook is independent of the original `pipeline2final.ipynb`. Do not run `Run All` on the original notebook.


## 1. Run configuration and imports

Extract the ZIP and open this notebook from the extracted project folder. The expected data path is `./stage4_data/`.

The seed list is read from the archived run manifest and cross-checked against the saved model-metric file. The five seeds are model-initialization/tuning seeds; the outer train/validation/test assignments remain frozen.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Mapping, Sequence, Any
import hashlib, json, platform, sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "stage4_data"
REFERENCE_DIR = DATA_DIR / "reference_splits"
OUTPUT_ROOT = PROJECT_DIR / "outputs"
RUN_ID = "stage4_5_part2_preprocessing_model_factory_v3"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID

N_BOOT = 2000
LATENT_DIM = 128
MODALITY_DROPOUT = 0.20
MC_PASSES = 30
MODEL_VARIANTS = ["EarlyFusion", "LateFusion", "AttentionFusion", "AUAF"]

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)
print("PyTorch:", torch.__version__)


## 2. Verify the frozen Stage 4.4 inputs

The SHA-256 values below identify the exact three task datasets and the feature manifest used in this revision package. A changed file stops execution.


In [ ]:
EXPECTED_SHA256 = {'diagnosis_binary_3700_asof.csv': '0f9d248921561a623eb9bc32108118a0e32c8970946e89500d75739cf61e13b8', 'diagnosis_multiclass_3700_asof.csv': 'e5b2ff31a7491383c0bdfe28376167db1e26dc95bce310359cc9ab4ebe9c5447', 'feature_manifest_34_predictors.csv': '489d81d536e94491db4e52fcd0f96b37b436c9f74663582e836008e9bbe34c76', 'mci_conversion_911_asof.csv': '369be50fd2ef6e57431de258c19dfa51fbea06f6825a6088bdc4390e4a4b361a'}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

missing_files = [name for name in EXPECTED_SHA256 if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing required inputs: {missing_files}")

ACTUAL_SHA256 = {name: sha256_file(DATA_DIR / name) for name in EXPECTED_SHA256}
for name, expected in EXPECTED_SHA256.items():
    if ACTUAL_SHA256[name] != expected:
        raise RuntimeError(f"SHA-256 mismatch: {name}")

print("Input SHA-256 verification: PASS")

## 3. Verify the original seeds and frozen split assignments

This section directly compares each Stage 4.4 dataset against the corresponding split manifest archived with the original results. It checks participant IDs, split labels, and target labels row by row.


In [ ]:
legacy_runtime = json.loads((REFERENCE_DIR / "legacy_runtime_manifest.json").read_text(encoding="utf-8"))
seed_evidence = json.loads((REFERENCE_DIR / "legacy_seed_evidence.json").read_text(encoding="utf-8"))

SEEDS = [int(x) for x in legacy_runtime["seeds"]]
assert SEEDS == [42, 123, 456, 789, 1024]
assert seed_evidence["all_model_metrics_uncalibrated_seeds"] == SEEDS
assert seed_evidence["verified_seed_list"] == SEEDS

SPLIT_REFERENCE = {
    "diagnosis_multiclass": ("subject_split_manifest__cn_mci_ad.csv", "target_multiclass", "TARGET_MULTICLASS"),
    "diagnosis_binary": ("subject_split_manifest__ad_vs_nonad.csv", "target_binary", "TARGET_BINARY"),
    "mci_conversion": ("subject_split_manifest__early_decline.csv", "target_decline", "TARGET_CONVERSION"),
}

print("Verified five seeds:", SEEDS)
print("Seed interpretation: model initialization/tuning seeds; not independent outer splits.")


## 4. Load task datasets and validate their structure

The three task-specific split assignments are retained exactly as supplied. The binary and multiclass diagnostic tasks are not forced to share a split assignment with each other.


In [ ]:
TASK_FILES = {
    "diagnosis_multiclass": ("diagnosis_multiclass_3700_asof.csv", "TARGET_MULTICLASS", 3700, 3),
    "diagnosis_binary": ("diagnosis_binary_3700_asof.csv", "TARGET_BINARY", 3700, 2),
    "mci_conversion": ("mci_conversion_911_asof.csv", "TARGET_CONVERSION", 911, 2),
}

TASKS = {}
for task_name, (filename, target_col, expected_n, n_classes) in TASK_FILES.items():
    frame = pd.read_csv(DATA_DIR / filename, parse_dates=["INDEX_DATE"])
    if len(frame) != expected_n:
        raise AssertionError(f"{task_name}: expected {expected_n} rows; found {len(frame)}")
    TASKS[task_name] = {
        "frame": frame,
        "target": target_col,
        "n_classes": n_classes,
        "filename": filename,
    }

FEATURE_MANIFEST = pd.read_csv(DATA_DIR / "feature_manifest_34_predictors.csv")
if FEATURE_MANIFEST["feature"].duplicated().any():
    raise AssertionError("Duplicate feature names in feature manifest.")

FEATURES = FEATURE_MANIFEST["feature"].astype(str).tolist()
CLINICAL_FEATURES = FEATURE_MANIFEST.loc[
    FEATURE_MANIFEST["modality"].eq("clinical_cognitive"), "feature"
].astype(str).tolist()
BIOMARKER_FEATURES = FEATURE_MANIFEST.loc[
    FEATURE_MANIFEST["modality"].eq("fluid_biomarker"), "feature"
].astype(str).tolist()

assert len(FEATURES) == 34
assert len(CLINICAL_FEATURES) == 31
assert len(BIOMARKER_FEATURES) == 3
assert set(CLINICAL_FEATURES).isdisjoint(BIOMARKER_FEATURES)
assert set(CLINICAL_FEATURES) | set(BIOMARKER_FEATURES) == set(FEATURES)
assert not ({"LANGUAGE_CODE", "APVOLUME"} & set(FEATURES))

REQUIRED_META = [
    "RID", "PHASE", "VISCODE", "INDEX_VISCODE2", "INDEX_DATE",
    "INDEX_DATE_SOURCE", "split", "ANY_MODEL_FEATURE_AVAILABLE_ASOF",
]
ALLOWED_SPLITS = {"train", "validation", "test"}

def validate_task(task_name: str, spec: dict) -> dict:
    frame, target = spec["frame"], spec["target"]
    required = set(REQUIRED_META + [target] + FEATURES)
    missing = sorted(required - set(frame.columns))
    if missing:
        raise AssertionError(f"{task_name}: missing columns {missing}")
    if frame["RID"].isna().any() or frame["RID"].duplicated().any():
        raise AssertionError(f"{task_name}: RID must be nonmissing and unique.")
    expected_missing_dates = 15 if task_name in {"diagnosis_multiclass", "diagnosis_binary"} else 0
    observed_missing_dates = int(frame["INDEX_DATE"].isna().sum())
    if observed_missing_dates != expected_missing_dates or frame[target].isna().any():
        raise AssertionError(
            f"{task_name}: expected {expected_missing_dates} missing Index Dates; "
            f"observed {observed_missing_dates}; target missing={bool(frame[target].isna().any())}."
        )
    if "INDEX_DATE_MISSING" not in frame.columns:
        raise AssertionError(f"{task_name}: INDEX_DATE_MISSING provenance flag is required.")
    if not frame["INDEX_DATE_MISSING"].astype(bool).equals(frame["INDEX_DATE"].isna()):
        raise AssertionError(f"{task_name}: Index Date missingness flag mismatch.")
    biomarker_cols = [c for c in FEATURES if c in BIOMARKER_FEATURES]
    if frame.loc[frame["INDEX_DATE"].isna(), biomarker_cols].notna().any(axis=None):
        raise AssertionError(f"{task_name}: biomarker values present without an Index Date.")
    if "BIOMARKER_AFTER_INDEX" in frame.columns:
        if frame.loc[frame["BIOMARKER_AFTER_INDEX"].astype(bool), biomarker_cols].notna().any(axis=None):
            raise AssertionError(f"{task_name}: post-index biomarker values remain.")
    if set(frame["split"].astype(str).str.lower().unique()) != ALLOWED_SPLITS:
        raise AssertionError(f"{task_name}: unexpected split labels.")
    if frame[FEATURES].isin([-1, -4]).to_numpy().any():
        raise AssertionError(f"{task_name}: ADNI missing sentinels remain.")
    if not all(pd.api.types.is_numeric_dtype(frame[c]) for c in FEATURES):
        raise AssertionError(f"{task_name}: expected numeric predictors only.")
    observed_any = frame[FEATURES].notna().any(axis=1)
    stored_any = frame["ANY_MODEL_FEATURE_AVAILABLE_ASOF"].astype(bool)
    if not observed_any.equals(stored_any):
        raise AssertionError(f"{task_name}: as-of availability flag mismatch.")
    split_sets = {
        s: set(frame.loc[frame["split"].astype(str).str.lower().eq(s), "RID"])
        for s in ALLOWED_SPLITS
    }
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        if split_sets[a] & split_sets[b]:
            raise AssertionError(f"{task_name}: participant overlap across {a}/{b}.")
    return {
        "task": task_name,
        "n": int(len(frame)),
        "class_counts": {str(k): int(v) for k, v in frame[target].value_counts().sort_index().items()},
        "split_counts": {str(k): int(v) for k, v in frame["split"].value_counts().items()},
        "unique_RID": int(frame["RID"].nunique()),
        "all_predictors_numeric": True,
        "no_ADNI_sentinels": True,
    }

VALIDATION_SUMMARY = pd.DataFrame([
    validate_task(name, spec) for name, spec in TASKS.items()
])
print(VALIDATION_SUMMARY.to_string(index=False))


## 5. Exact comparison with the archived split manifests

A mismatch in any participant, split label, or target label stops the notebook. No split is regenerated.


In [ ]:
split_check_rows = []
for task_name, (reference_name, reference_target, current_target) in SPLIT_REFERENCE.items():
    ref_path = REFERENCE_DIR / reference_name
    if not ref_path.is_file():
        raise FileNotFoundError(ref_path)
    ref = pd.read_csv(ref_path).rename(columns={"_split": "split"})
    cur = TASKS[task_name]["frame"][["RID", "split", current_target]].copy()
    merged = ref[["RID", "split", reference_target]].merge(
        cur, on="RID", how="outer", suffixes=("_archived", "_current"),
        indicator=True, validate="one_to_one"
    )
    if not merged["_merge"].eq("both").all():
        raise AssertionError(f"{task_name}: participant roster differs from archived split manifest.")
    split_match = merged["split_archived"].astype(str).str.lower().eq(
        merged["split_current"].astype(str).str.lower()
    )
    target_match = pd.to_numeric(merged[reference_target], errors="raise").eq(
        pd.to_numeric(merged[current_target], errors="raise")
    )
    if not split_match.all():
        raise AssertionError(f"{task_name}: split assignments differ from archived manifest.")
    if not target_match.all():
        raise AssertionError(f"{task_name}: target labels differ from archived manifest.")
    split_check_rows.append({
        "task": task_name,
        "participants_checked": int(len(merged)),
        "RID_match_pct": 100.0,
        "split_match_pct": 100.0,
        "target_match_pct": 100.0,
        "status": "PASS",
    })

SPLIT_CHECK = pd.DataFrame(split_check_rows)
print(SPLIT_CHECK.to_string(index=False))


## 6. Prespecified feature groups and reviewer sensitivity sets

The source manifest contains 34 candidate predictors. Two demographic fields (`PTENGSPKAGE`, `PTIMMAGE`) remain unverified at the exact phase-specific ADNI DATADIC level and are excluded from every training feature set until their official definitions and coding are confirmed. The verified candidate set therefore contains 32 predictors: 29 clinical/cognitive fields and 3 fluid biomarkers. The two excluded fields remain documented in the source inventory and audit outputs.


In [ ]:
UNVERIFIED_FIELDS = ["PTENGSPKAGE", "PTIMMAGE"]
VERIFIED_FEATURES = [f for f in FEATURES if f not in UNVERIFIED_FIELDS]

DEMOGRAPHIC = ["PTGENDER", "PTEDUCAT"]
MMSE = ["MMSEASON"]
MOCA = ["TRAILS", "CLOCKCON", "CLOCKNO", "CLOCKHAN", "FFLUENCY", "MOCA"]
FAQ = ["FAQFINAN", "FAQFORM", "FAQSHOP", "FAQGAME", "FAQBEVG", "FAQMEAL",
       "FAQEVENT", "FAQTV", "FAQREM", "FAQTRAVL", "FAQTOTAL"]
CDR = ["CDMEMORY", "CDRSB"]
NEUROBAT = ["CLOCKCIRC", "CLOCKSYM", "CLOCKNUM", "CLOCKHAND", "CLOCKTIME", "CLOCKSCOR", "DIGITSCOR"]
BIOMARKERS = ["ABETA40", "ABETA42", "PTAU"]

assert len(FEATURES) == 34
assert len(VERIFIED_FEATURES) == 32
assert len(CLINICAL_FEATURES) == 31
assert len(BIOMARKER_FEATURES) == 3
assert len([f for f in VERIFIED_FEATURES if f in CLINICAL_FEATURES]) == 29
assert set(DEMOGRAPHIC + MMSE + MOCA + FAQ + CDR + NEUROBAT + BIOMARKERS) == set(VERIFIED_FEATURES)

FEATURE_SETS = {
    "S0_verified_asof": VERIFIED_FEATURES.copy(),
    "S1_minus_CDR": [f for f in VERIFIED_FEATURES if f not in CDR],
    "S2_minus_FAQ": [f for f in VERIFIED_FEATURES if f not in FAQ],
    "S3_minus_CDR_FAQ": [f for f in VERIFIED_FEATURES if f not in CDR + FAQ],
    "S4_minus_MMSE_MOCA": [f for f in VERIFIED_FEATURES if f not in MMSE + MOCA],
    "S5_strict_CDR_FAQ_MMSE_MOCA": [
        f for f in VERIFIED_FEATURES if f not in CDR + FAQ + MMSE + MOCA
    ],
    "S6_demographic_biomarker_only": DEMOGRAPHIC + BIOMARKERS,
}

FEATURE_SET_TABLE = pd.DataFrame([
    {
        "feature_set": name,
        "candidate_predictor_count": len(cols),
        "removed_features": ";".join([f for f in FEATURES if f not in cols]),
        "retained_features": ";".join(cols),
        "analysis_role": (
            "Exploratory reference for conversion; not evidence of multimodal fusion"
            if name == "S6_demographic_biomarker_only"
            else (
                "Primary verified as-of-index candidate"
                if name == "S0_verified_asof"
                else "Prespecified circularity/severity sensitivity set"
            )
        ),
        "status": "Candidate — not yet trained",
    }
    for name, cols in FEATURE_SETS.items()
])

definition_hold = FEATURE_MANIFEST[FEATURE_MANIFEST["feature"].isin(UNVERIFIED_FIELDS)][
    ["feature", "definition_status", "circularity_risk", "risk_reason_and_action"]
].copy()
assert set(definition_hold["feature"]) == set(UNVERIFIED_FIELDS)

print(FEATURE_SET_TABLE[["feature_set", "candidate_predictor_count", "analysis_role"]].to_string(index=False))
print("\nUnverified ADNI fields excluded from all training feature sets:")
print(definition_hold.to_string(index=False))


## 7. Train-only numeric preprocessing

All 34 current candidate predictors are numeric in the frozen task files. The preprocessor below:

- estimates medians, means, and scales from the training split only;
- keeps a stable feature order;
- if a feature is entirely missing in training, disables it for every split rather than allowing test-only values into the model;
- computes modality availability from the effective features actually usable by the training split.

No validation or test statistics are used to fit preprocessing.


In [ ]:
class TrainOnlyNumericPreprocessor:
    def __init__(self, columns: Sequence[str]):
        self.columns = list(columns)
        self.medians_ = None
        self.means_ = None
        self.scales_ = None
        self.all_missing_train_ = None
        self.fit_rids_ = None
        self.fit_n_ = None

    @staticmethod
    def _array(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        values = frame[list(columns)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
        values[~np.isfinite(values)] = np.nan
        return values

    def fit(self, frame: pd.DataFrame, train_mask: np.ndarray):
        train_mask = np.asarray(train_mask, dtype=bool)
        if train_mask.sum() == 0:
            raise ValueError("Cannot fit preprocessing: empty training split.")
        x_train = self._array(frame.loc[train_mask], self.columns)

        medians = np.zeros(x_train.shape[1], dtype=np.float64)
        all_missing = np.zeros(x_train.shape[1], dtype=bool)
        for j in range(x_train.shape[1]):
            observed = x_train[:, j][np.isfinite(x_train[:, j])]
            if observed.size == 0:
                all_missing[j] = True
                medians[j] = 0.0
            else:
                medians[j] = float(np.median(observed))

        # Impute with training medians.
        x_imp = np.where(np.isfinite(x_train), x_train, medians[None, :])
        means = x_imp.mean(axis=0)
        scales = x_imp.std(axis=0, ddof=0)
        scales[~np.isfinite(scales) | (scales < 1e-12)] = 1.0

        self.medians_ = medians
        self.means_ = means
        self.scales_ = scales
        self.all_missing_train_ = all_missing
        self.fit_rids_ = sorted(frame.loc[train_mask, "RID"].astype(int).tolist())
        self.fit_n_ = int(train_mask.sum())
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if self.medians_ is None:
            raise RuntimeError("Preprocessor has not been fitted.")
        x = self._array(frame, self.columns)
        # Do not allow features absent in training to contribute test-only information.
        x[:, self.all_missing_train_] = np.nan
        x_imp = np.where(np.isfinite(x), x, self.medians_[None, :])
        z = (x_imp - self.means_[None, :]) / self.scales_[None, :]
        z[~np.isfinite(z)] = 0.0
        return z.astype(np.float32)

    def effective_columns(self) -> List[str]:
        return [c for c, missing in zip(self.columns, self.all_missing_train_) if not missing]

    def audit(self) -> dict:
        return {
            "columns_requested": self.columns,
            "columns_effective": self.effective_columns(),
            "all_missing_in_train": [
                c for c, missing in zip(self.columns, self.all_missing_train_) if missing
            ],
            "fit_n": self.fit_n_,
            "fit_RID_count": len(self.fit_rids_),
            "fit_RID_sha256": hashlib.sha256(
                ",".join(map(str, self.fit_rids_)).encode("utf-8")
            ).hexdigest(),
        }


@dataclass
class PreparedTask:
    name: str
    feature_set: str
    frame: pd.DataFrame
    target: str
    n_classes: int
    modalities: List[str]
    features_by_modality: Dict[str, List[str]]
    matrices: Dict[str, np.ndarray]
    masks: np.ndarray
    processors: Dict[str, TrainOnlyNumericPreprocessor]
    fit_rids: List[int]

    def idx(self, split: str, eligible_only: bool = False) -> np.ndarray:
        split_mask = self.frame["split"].astype(str).str.lower().eq(split).to_numpy()
        idx = np.flatnonzero(split_mask)
        if eligible_only:
            idx = idx[self.masks[idx].sum(axis=1) > 0]
        return idx

    def paired_modality_idx(self, split: str) -> np.ndarray:
        idx = self.idx(split, eligible_only=False)
        if self.masks.shape[1] < 2:
            return np.array([], dtype=int)
        return idx[(self.masks[idx] > 0).all(axis=1)]


MODALITY_ORDER = ["clinical_cognitive", "fluid_biomarker"]

def prepare_task(task_name: str, feature_set_name: str) -> PreparedTask:
    spec = TASKS[task_name]
    frame = spec["frame"].reset_index(drop=True).copy()
    selected = FEATURE_SETS[feature_set_name]
    train_mask = frame["split"].astype(str).str.lower().eq("train").to_numpy()

    features_by_modality = {}
    matrices = {}
    processors = {}
    mask_columns = []
    active_modalities = []
    all_fit_rids = None

    for modality in MODALITY_ORDER:
        requested = [
            f for f in selected
            if f in set(FEATURE_MANIFEST.loc[
                FEATURE_MANIFEST["modality"].eq(modality), "feature"
            ].astype(str))
        ]
        if not requested:
            continue

        processor = TrainOnlyNumericPreprocessor(requested).fit(frame, train_mask)
        effective = processor.effective_columns()
        if not effective:
            continue

        # Refit on effective columns only, still using training participants only.
        processor = TrainOnlyNumericPreprocessor(effective).fit(frame, train_mask)
        values = processor.transform(frame)
        raw_available = frame[effective].notna().any(axis=1).to_numpy()

        features_by_modality[modality] = effective
        matrices[modality] = values
        processors[modality] = processor
        mask_columns.append(raw_available.astype(np.float32))
        active_modalities.append(modality)

        current_fit_rids = processor.fit_rids_
        if all_fit_rids is None:
            all_fit_rids = current_fit_rids
        elif current_fit_rids != all_fit_rids:
            raise AssertionError("Preprocessors must be fitted on the same training participants.")

    if not active_modalities:
        raise RuntimeError(f"{task_name}/{feature_set_name}: no train-usable modality.")

    masks = np.column_stack(mask_columns).astype(np.float32)
    if not np.isfinite(masks).all():
        raise AssertionError("Non-finite modality mask.")
    for modality, values in matrices.items():
        if values.shape[0] != len(frame) or not np.isfinite(values).all():
            raise AssertionError(f"Invalid transformed matrix for {task_name}/{feature_set_name}/{modality}.")

    return PreparedTask(
        name=task_name,
        feature_set=feature_set_name,
        frame=frame,
        target=spec["target"],
        n_classes=spec["n_classes"],
        modalities=active_modalities,
        features_by_modality=features_by_modality,
        matrices=matrices,
        masks=masks,
        processors=processors,
        fit_rids=all_fit_rids or [],
    )


PREPARED = {
    (task_name, feature_set_name): prepare_task(task_name, feature_set_name)
    for task_name in TASKS
    for feature_set_name in FEATURE_SETS
}
print("Prepared task/feature-set combinations:", len(PREPARED))

preprocessor_fit_rows = []
for (task_name, feature_set_name), pack in PREPARED.items():
    expected_train_rids = set(
        pack.frame.loc[
            pack.frame["split"].astype(str).str.lower().eq("train"), "RID"
        ].astype(int)
    )
    for modality, processor in pack.processors.items():
        fitted_rids = set(processor.fit_rids_)
        if fitted_rids != expected_train_rids:
            raise AssertionError(
                f"{task_name}/{feature_set_name}/{modality}: preprocessor fit IDs are not exactly train IDs."
            )
        preprocessor_fit_rows.append({
            "task": task_name,
            "feature_set": feature_set_name,
            "modality": modality,
            "fit_participant_n": processor.fit_n_,
            "expected_train_participant_n": len(expected_train_rids),
            "fit_RID_exactly_train_RID": True,
            "all_missing_in_train_features": ";".join(processor.audit()["all_missing_in_train"]),
            "effective_feature_n": len(processor.effective_columns()),
        })

PREPROCESSOR_FIT_AUDIT = pd.DataFrame(preprocessor_fit_rows)
print("Train-only preprocessing fit audit: PASS", len(PREPROCESSOR_FIT_AUDIT), "modality pipelines")



## 8. Participant availability and matched-comparison audit

The broad candidate set and every sensitivity set are assessed on the frozen splits. For pairwise comparisons, the reference model and its reduced-feature counterpart must be evaluated on the same eligible test participants.

The complete-modality subset is also counted separately for the controlled biomarker-missingness experiment. Its size is a design limitation that must be reported, not hidden.


In [ ]:
eligibility_rows = []
matched_rows = []
overlap_rows = []

for task_name, spec in TASKS.items():
    frame = spec["frame"].reset_index(drop=True)
    target = spec["target"]
    prepared_s0 = PREPARED[(task_name, "S0_verified_asof")]

    for feature_set_name in FEATURE_SETS:
        pack = PREPARED[(task_name, feature_set_name)]
        for split in ["train", "validation", "test"]:
            all_idx = pack.idx(split, eligible_only=False)
            eligible_idx = pack.idx(split, eligible_only=True)
            y = frame.iloc[eligible_idx][target]
            eligibility_rows.append({
                "task": task_name,
                "feature_set": feature_set_name,
                "split": split,
                "split_n": int(len(all_idx)),
                "eligible_n": int(len(eligible_idx)),
                "no_usable_predictor_n": int(len(all_idx) - len(eligible_idx)),
                "eligible_class_counts": json.dumps(
                    {str(k): int(v) for k, v in y.value_counts().sort_index().items()}
                ),
                "effective_clinical_features": len(pack.features_by_modality.get("clinical_cognitive", [])),
                "effective_biomarker_features": len(pack.features_by_modality.get("fluid_biomarker", [])),
            })

        # Pairwise S0 versus Sx: use the reduced-set eligible test IDs for both conditions.
        s0_test = set(frame.loc[
            prepared_s0.idx("test", eligible_only=True), "RID"
        ].astype(int))
        sx_test = set(frame.loc[
            pack.idx("test", eligible_only=True), "RID"
        ].astype(int))
        common = s0_test & sx_test
        if common != sx_test:
            raise AssertionError("Feature sets are expected to be nested subsets of S0.")
        matched_rows.append({
            "task": task_name,
            "comparison": f"S0_verified_asof vs {feature_set_name}",
            "matched_test_RID_n": len(common),
            "reduced_set_eligible_test_n": len(sx_test),
            "same_test_participants_for_pair": True,
            "note": "Pairwise matched test population; do not compare against unmatched S0 metrics.",
        })

    # Modality overlap for S0 and S6, based on train-usable features only.
    for feature_set_name in ["S0_verified_asof", "S6_demographic_biomarker_only"]:
        overlap_pack = PREPARED[(task_name, feature_set_name)]
        clinical_idx = overlap_pack.modalities.index("clinical_cognitive")
        biomarker_idx = overlap_pack.modalities.index("fluid_biomarker")
        for split in ["train", "validation", "test"]:
            idx = overlap_pack.idx(split, eligible_only=False)
            clinical_present = overlap_pack.masks[idx, clinical_idx] > 0
            biomarker_present = overlap_pack.masks[idx, biomarker_idx] > 0
            overlap_rows.append({
                "task": task_name,
                "feature_set": feature_set_name,
                "split": split,
                "clinical_available_n": int(clinical_present.sum()),
                "biomarker_available_n": int(biomarker_present.sum()),
                "both_modalities_available_n": int((clinical_present & biomarker_present).sum()),
                "clinical_only_n": int((clinical_present & ~biomarker_present).sum()),
                "biomarker_only_n": int((~clinical_present & biomarker_present).sum()),
                "neither_n": int((~clinical_present & ~biomarker_present).sum()),
            })

ELIGIBILITY_SUMMARY = pd.DataFrame(eligibility_rows)
MATCHED_TEST_SUMMARY = pd.DataFrame(matched_rows)
MODALITY_OVERLAP_SUMMARY = pd.DataFrame(overlap_rows)

print("Eligibility summary rows:", len(ELIGIBILITY_SUMMARY))
print("\nS0 modality overlap:")
print(MODALITY_OVERLAP_SUMMARY.to_string(index=False))
print("\nPairwise matched test cohorts:")
print(MATCHED_TEST_SUMMARY[MATCHED_TEST_SUMMARY["feature_set" if "feature_set" in MATCHED_TEST_SUMMARY.columns else "comparison"].notna()].head(10).to_string(index=False))


## 9. Canonical neural model factory

The four neural variants preserve the study's intended comparison: EarlyFusion, LateFusion, AttentionFusion, and AUAF.

A participant with no available modality is not passed silently to a fusion model. The model factory raises an explicit error for an all-zero modality mask. This prevents meaningless predictions and avoids NaNs in attention layers.


The AUAF implementation retains the original per-modality mask-token parameterization and the original 0.20 structured modality-dropout rate. MC-dropout inference is implemented separately so that stochastic dropout is active without accidentally reactivating structured modality dropout.


The AUAF implementation retains the original per-modality mask-token parameterization and 0.20 structured modality-dropout rate. MC-dropout inference enables only `nn.Dropout` layers while keeping the parent model in `eval()` mode, so structured modality dropout is not accidentally reactivated.

In [ ]:
from auaf_model_factory import (
    LATENT_DIM as MODEL_LATENT_DIM,
    MODALITY_DROPOUT as MODEL_MODALITY_DROPOUT,
    MC_PASSES as MODEL_MC_PASSES,
    require_at_least_one_modality,
    ResidualMLPBlock,
    ModalityEncoder,
    EarlyFusionMLP,
    LateFusionNetwork,
    CrossModalAttentionBlock,
    MissingAwareAttentionFusion,
    ModalityGatingNetwork,
    CrossModalTransformerBlock,
    AdaptiveUncertaintyAwareFusion,
    build_model,
)

assert LATENT_DIM == MODEL_LATENT_DIM
assert MODALITY_DROPOUT == MODEL_MODALITY_DROPOUT
assert MC_PASSES == MODEL_MC_PASSES
MODEL_FACTORY_SHA256 = sha256_file(PROJECT_DIR / "auaf_model_factory.py")
print("Canonical model factory SHA-256:", MODEL_FACTORY_SHA256)


## 10. Model-factory smoke tests (no training)

The smoke tests check finite forward passes for:
- both modalities available;
- clinical modality only;
- fluid-biomarker modality only.

An all-zero modality mask must raise an explicit error. No optimizer step, tuning, checkpoint, or model training is run here.


In [ ]:
torch.manual_seed(2026)
smoke_rows = []

for task_name, spec in TASKS.items():
    pack = PREPARED[(task_name, "S0_verified_asof")]
    dimensions = {
        modality: int(pack.matrices[modality].shape[1])
        for modality in pack.modalities
    }
    if set(dimensions) != set(MODALITY_ORDER):
        raise AssertionError(f"{task_name}: expected both candidate modalities, found {list(dimensions)}")

    batch_size = 4
    inputs = {
        modality: torch.randn(batch_size, dimension, dtype=torch.float32)
        for modality, dimension in dimensions.items()
    }
    masks = torch.tensor([
        [1.0, 1.0],
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ], dtype=torch.float32)

    for variant in MODEL_VARIANTS:
        model = build_model(variant, dimensions, spec["n_classes"])
        for mode in ["eval", "train"]:
            model.train(mode == "train")
            with torch.set_grad_enabled(mode == "train"):
                logits = model(inputs, masks)
            finite = bool(torch.isfinite(logits).all().item())
            correct_shape = tuple(logits.shape) == (batch_size, spec["n_classes"])
            if not finite or not correct_shape:
                raise AssertionError(f"{task_name}/{variant}/{mode}: forward smoke test failed.")

            smoke_rows.append({
                "task": task_name,
                "model": variant,
                "mode": mode,
                "output_shape": str(tuple(logits.shape)),
                "finite_logits": finite,
                "both_modalities_pass": True,
                "clinical_only_pass": True,
                "biomarker_only_pass": True,
                "training_performed": False,
            })

        model.eval()
        all_zero_rejected = False
        try:
            with torch.no_grad():
                model(inputs, torch.zeros_like(masks))
        except ValueError:
            all_zero_rejected = True
        if not all_zero_rejected:
            raise AssertionError(f"{task_name}/{variant}: all-zero mask was not rejected.")

        smoke_rows[-1]["all_zero_mask_rejected"] = all_zero_rejected
        smoke_rows[-2]["all_zero_mask_rejected"] = all_zero_rejected

        if variant == "AUAF":
            model.eval()
            mc_result = model.mc_forward(inputs, masks, n_passes=3)
            mc_ok = (
                model.training is False
                and np.isfinite(mc_result["mean_prob"]).all()
                and np.isfinite(mc_result["std_prob"]).all()
                and np.isfinite(mc_result["entropy"]).all()
                and np.isfinite(mc_result["mean_gates"]).all()
            )
            if not mc_ok:
                raise AssertionError(f"{task_name}/AUAF: safe MC-dropout smoke test failed.")
            smoke_rows[-2]["mc_dropout_without_modality_dropout_pass"] = True
            smoke_rows[-1]["mc_dropout_without_modality_dropout_pass"] = None

MODEL_FACTORY_SMOKE_TEST = pd.DataFrame(smoke_rows)
print(MODEL_FACTORY_SMOKE_TEST.groupby(["model", "mode"])[["finite_logits"]].all())
print("Smoke tests passed:", len(MODEL_FACTORY_SMOKE_TEST), "of 24 mode/task/model checks")


## 11. Save the deterministic audit outputs

This cell writes only small audit tables and a run manifest. It does not save or overwrite model checkpoints. Re-running with the same `RUN_ID` is allowed only when the output content is identical.


In [ ]:
def write_text_once(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != content:
            raise RuntimeError(f"Refusing to overwrite different existing output: {path}")
        return "EXISTING_IDENTICAL"
    path.write_text(content, encoding="utf-8")
    return "CREATED"

def write_csv_once(path: Path, frame: pd.DataFrame):
    content = frame.to_csv(index=False)
    return write_text_once(path, content)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_status = {}
output_status["split_manifest_checks.csv"] = write_csv_once(
    OUTPUT_DIR / "split_manifest_checks.csv", SPLIT_CHECK
)
output_status["preprocessor_fit_audit.csv"] = write_csv_once(
    OUTPUT_DIR / "preprocessor_fit_audit.csv", PREPROCESSOR_FIT_AUDIT
)
output_status["feature_set_registry.csv"] = write_csv_once(
    OUTPUT_DIR / "feature_set_registry.csv", FEATURE_SET_TABLE
)
output_status["feature_set_eligibility.csv"] = write_csv_once(
    OUTPUT_DIR / "feature_set_eligibility.csv", ELIGIBILITY_SUMMARY
)
output_status["matched_test_cohort_summary.csv"] = write_csv_once(
    OUTPUT_DIR / "matched_test_cohort_summary.csv", MATCHED_TEST_SUMMARY
)
output_status["modality_overlap_summary.csv"] = write_csv_once(
    OUTPUT_DIR / "modality_overlap_summary.csv", MODALITY_OVERLAP_SUMMARY
)
output_status["model_factory_smoke_test.csv"] = write_csv_once(
    OUTPUT_DIR / "model_factory_smoke_test.csv", MODEL_FACTORY_SMOKE_TEST
)

manifest_payload = {
    "run_id": RUN_ID,
    "stage": "4.5_part2_preprocessing_and_model_factory",
    "input_sha256": ACTUAL_SHA256,
    "model_factory_sha256": MODEL_FACTORY_SHA256,
    "seeds_verified": SEEDS,
    "bootstrap_resamples_planned": N_BOOT,
    "tasks": VALIDATION_SUMMARY.to_dict(orient="records"),
    "split_manifest_checks": SPLIT_CHECK.to_dict(orient="records"),
    "preprocessor_fit_audit_rows": int(len(PREPROCESSOR_FIT_AUDIT)),
    "feature_set_names": list(FEATURE_SETS),
    "model_variants_smoke_tested": MODEL_VARIANTS,
    "model_training_performed": False,
    "checkpoint_written": False,
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "torch_version": torch.__version__,
    "output_files": sorted(output_status.keys()),
}
manifest_text = json.dumps(manifest_payload, indent=2, sort_keys=True)
output_status["preprocessing_model_factory_manifest.json"] = write_text_once(
    OUTPUT_DIR / "preprocessing_model_factory_manifest.json", manifest_text
)

print("Output status:", output_status)
print("PART 2 PREPROCESSING AND MODEL-FACTORY CHECKS: PASS")
print("No model training or tuning was performed.")


## 12. Scientific guardrails and next part

1. `S6_demographic_biomarker_only` is an exploratory low-circularity reference. In the current diagnostic cohorts, the demographic and fluid-biomarker subsets do not co-occur in the same participants; it must not be used to claim a fusion advantage.
2. The S0 complete-modality test subgroup is limited. The current as-of-index files contain 80 multiclass-diagnosis, 67 binary-diagnosis, and 28 MCI-conversion test participants with both modalities available. Controlled missingness results must report these denominators and uncertainty intervals.
3. `PTENGSPKAGE` and `PTIMMAGE` remain on hold pending exact phase-specific ADNI DATADIC verification. They are not silently redefined here.
4. The five seeds are `[42, 123, 456, 789, 1024]`. The archived train/validation/test splits are unchanged.
5. No performance result is produced in this part.

**Next part:** implement the actual training runner with immutable run IDs, feature-set-aware preprocessing, fixed participant cohorts, validated checkpoint resume, and no silent row dropping. Then run the reviewer-requested circularity, controlled biomarker-missingness, and component-ablation experiments.
